# CS 521 HW2 — Problem 5: Certifier Design in ConstraintFlow

ConstraintFlow is a DSL for DNN certifiers based on abstract interpretation. This notebook
covers all four parts:

* **(a)** Design a new certifier different from DeepPoly.
* **(b)** Prove it sound with `provesound`.
* **(c)** Run DeepPoly and the new certifier on `mnist_relu_3_100.onnx` (first 100 test images,
  ε ∈ {0.005, 0.01, 0.02, 0.03, 0.05}) and compare.
* **(d)** Write and prove a `HardTanh` transformer in the DeepPoly shape.

**Starter code**: https://github.com/nalinwadhwa02/CCS521FA26HW/tree/main/hw2/q5
**ConstraintFlow**: https://github.com/uiuc-focal-lab/constraintflow (pinned commit `881e270`).

---
## Environment setup

```bash
python3.12 -m venv cfenv            # Python 3.10+ ; 3.12 used here (onnx/torch wheels)
source cfenv/bin/activate
git clone https://github.com/uiuc-focal-lab/constraintflow.git
cd constraintflow && git checkout 881e270
pip install "setuptools<81" -e .    # setuptools<81 so z3 can import pkg_resources
cd .. && pip install onnx torch torchvision
```

Then place `deeppoly.cf`, `run_cf.py`, `mnist_relu_3_100.onnx` (from the starter repo) and the
new `.cf` files in this directory. All cells below shell out to `./cfenv/bin/constraintflow`
and `./cfenv/bin/python run_cf.py`.

In [1]:
import subprocess, os

CFBIN = os.path.join(os.getcwd(), "cfenv", "bin")

def sh(args):
    """Run a command and print its output with ConstraintFlow's pkg_resources warning
    and run_cf.py's progress-percentage lines stripped out."""
    p = subprocess.run(args, capture_output=True, text=True)
    out = []
    for ln in (p.stdout + p.stderr).splitlines():
        low = ln.lower()
        if "pkg_resources" in low or "userwarning" in low or "import pkg" in low:
            continue
        s = ln.strip()
        if s and all(c in "0123456789.% " for c in s) and s.endswith("%"):
            continue  # drop MNIST download / compile progress bars
        out.append(ln)
    print("\n".join(out).strip())

def provesound(cf):   sh([os.path.join(CFBIN, "constraintflow"), "provesound", cf])
def run_cert(cf, extra=None):
    args = [os.path.join(CFBIN, "python"), "run_cf.py", cf,
            "--num-images", "100", "--eps", "0.005", "0.01", "0.02", "0.03", "0.05"]
    sh(args + (extra or []))

print("ConstraintFlow CLI:", os.path.exists(os.path.join(CFBIN, "constraintflow")))

ConstraintFlow CLI: True


---
## (a) A new certifier: DeepPoly refined with interval analysis (`dp2.cf`)

**What DeepPoly does.** DeepPoly keeps, per neuron, two concrete bounds `l,u` and two
polyhedral bounds `L,U`. The Affine transformer computes `l,u` by back-substituting the
affine expression all the way to the input layer; the ReLU transformer uses the DeepPoly
relaxation and, for a crossing ReLU, picks the lower bound (`0` or `x`) by the area heuristic.

**What is new in `dp2`.** Back-substitution is relational and usually tight, but on a
crossing-ReLU network the repeated triangle relaxations can occasionally make a
back-substituted bound *looser* than a single **interval (box)** step taken from the
already-tight concrete bounds of the previous layer. `dp2` computes **both** at every Affine
layer and keeps the better one:

$$ l = \max(\text{interval\_lower},\ \text{backsub\_lower}), \qquad
   u = \min(\text{interval\_upper},\ \text{backsub\_upper}). $$

The interval bound is the one-step evaluation (`.map`) of the affine expression against
`prev[l], prev[u]`; the back-substituted bound is DeepPoly's. The max/min of two sound
lower/upper bounds is sound, and because one of the two is *exactly* DeepPoly's, **`dp2` is
never less precise than DeepPoly.** It stays in the same `(l,u,L,U)` shape, so no new shape
members are needed. (This follows `polyz.cf`'s `max(box, backsub)` pattern from the
ConstraintFlow examples.)

**Why it is useful.** It is a cheap, guaranteed refinement that shows how a relational domain
(DeepPoly) and a non-relational domain (intervals) can be combined soundly, and it can only
help, never hurt.

The `dp2.cf` source:

In [2]:
print(open("dp2.cf").read())

// dp2.cf : DeepPoly refined with interval (box) analysis  -- "DeepPoly x Interval".
//
// What is new compared to deeppoly.cf
// -----------------------------------
// DeepPoly computes each neuron's concrete bounds l,u purely by polyhedral
// back-substitution of the affine expression down to the input layer. That is
// relational and usually tight, but on a crossing-ReLU network the repeated
// triangle relaxations can occasionally make a back-substituted bound LOOSER
// than a single interval (box) step taken from the already-tight concrete
// bounds of the immediately previous layer.
//
// dp2 runs BOTH at every Affine layer and keeps the better one:
//     l = max( interval_lower , backsubstituted_lower )
//     u = min( interval_upper , backsubstituted_upper )
// The interval bound is the one-step evaluation (.map) of the affine expression
// against prev[l],prev[u]; the back-substituted bound is DeepPoly's. Taking the
// max/min of two sound lower/upper bounds is sound, and bec

### A second design that matches the hint: keep *both* ReLU lower bounds (`dp2_dualbound.cf`)

The hint suggests keeping **both** sound lower bounds (`0` and `x`) for each crossing ReLU and
back-substituting through both chains, taking the max. I implemented this too
(`dp2_dualbound.cf`): a 5-member shape `(l,u,L,U,La)` where `L` is DeepPoly's area-heuristic
choice and `La` is the opposite choice, with
`l = max(backsub via L, backsub via La)`. Because one chain is DeepPoly's, it is sound and
provably ≥ DeepPoly — and **`provesound` proves it sound** (shown in part (b)).

However, the ConstraintFlow compiler at the pinned commit `881e270` parallelizes the two
same-direction (both-lower) back-substitution traversals into threads and emits a faulty
`nonlocal`, so it raises `UnboundLocalError: ... 'phi_trav_exp1_1_1' ...` at *run* time (the
compiler supports exactly one lower + one upper traversal, as in DeepPoly). The design is
therefore proven sound but cannot be executed with the stock `run_cf.py`; the runnable
comparison in part (c) uses the interval-refined `dp2.cf`, which is also sound and also
≥ DeepPoly.

---
## (b) Soundness with `provesound`

Expected: `Proved Affine` and `Proved Relu`.

In [3]:
provesound("dp2.cf")

Proved Affine
Proved Relu
Provesound result: None


`provesound` on the proven-sound dual-lower-bound design (`dp2_dualbound.cf`):

In [4]:
provesound("dp2_dualbound.cf")

Proved Affine
Proved Relu
Provesound result: None


---
## (c) Running DeepPoly and `dp2` on the network

First 100 MNIST test images, ε ∈ {0.005, 0.01, 0.02, 0.03, 0.05}, on `mnist_relu_3_100.onnx`
(784-100-100-100-10, ReLU, clean accuracy 97.25%). An image is *verified* at ε if the
certifier proves the true-label logit exceeds every other logit on the whole L∞ ball.

```bash
python run_cf.py deeppoly.cf --num-images 100 --eps 0.005 0.01 0.02 0.03 0.05
python run_cf.py dp2.cf      --num-images 100 --eps 0.005 0.01 0.02 0.03 0.05
```

In [5]:
print("===== DeepPoly =====")
run_cert("deeppoly.cf")

===== DeepPoly =====


Network: mnist_relu_3_100.onnx
Clean accuracy on 100 images: 99.00%
     eps   verified   time/img (s)
   0.005     99.00%         0.2234
    0.01     94.00%         0.2367
    0.02     83.00%         0.2258
    0.03     49.00%         0.2243
    0.05      2.00%         0.2238


In [6]:
print("===== dp2 (DeepPoly + interval refinement) =====")
run_cert("dp2.cf")

===== dp2 (DeepPoly + interval refinement) =====


Network: mnist_relu_3_100.onnx
Clean accuracy on 100 images: 99.00%
     eps   verified   time/img (s)
   0.005     99.00%         0.2437
    0.01     94.00%         0.2419
    0.02     83.00%         0.2423
    0.03     49.00%         0.2443
    0.05      2.00%         0.2438


### Comparison

| ε | DeepPoly verified | dp2 verified |
|------|------|------|
| 0.005 | 99% | 99% |
| 0.01  | 94% | 94% |
| 0.02  | 83% | 83% |
| 0.03  | 49% | 49% |
| 0.05  | 2%  | 2%  |

**The results are identical. Why?** For this feed-forward ReLU network, DeepPoly's
back-substituted bound is, at every Affine layer, always at least as tight as the one-step
interval bound computed from the previous layer's concrete `[l,u]`. Back-substitution keeps
the symbolic affine relationship down to the input and only then concretizes, whereas the
interval step concretizes immediately at the previous layer and loses all cross-neuron
correlation. Hence `max(interval_lower, backsub_lower) = backsub_lower` and
`min(interval_upper, backsub_upper) = backsub_upper` for every neuron, so `dp2` reduces
*exactly* to DeepPoly and verifies the same images. `dp2` is still a sound, valid certifier
that is **guaranteed never worse** than DeepPoly; it simply cannot do *better* here because
DeepPoly's relational bounds already dominate the interval bounds on this architecture.

To do strictly better one must carry *more* information per neuron — e.g. the dual
lower-bound design `dp2_dualbound.cf` (proven sound above), which the current compiler cannot
execute, or a richer abstract domain (a zonotope alongside the polyhedral bounds). Both are
genuinely more precise in principle; interval refinement is not, on a feed-forward net.

---
## (d) A `HardTanh` transformer in the DeepPoly shape

`HardTanh(x) = max(-1, min(1, x))` is **piecewise linear** and **monotone increasing**, so the
concrete bounds are `l = HardTanh(prev[l])`, `u = HardTanh(prev[u])`. For the polyhedral bounds
`L, U` we give sound linear bounds over `[prev[l], prev[u]]` by cases (`pl=prev[l]`, `pu=prev[u]`):

| case | condition | shape = `(l, u, L, U)` |
|---|---|---|
| flat-low   | `pu ≤ -1` | `(-1, -1, -1, -1)` |
| flat-high  | `pl ≥ 1`  | `(1, 1, 1, 1)` |
| linear     | `pl ≥ -1 ∧ pu ≤ 1` | `(pl, pu, x, x)` |
| upper knee | `pl ≥ -1 ∧ pu > 1` (concave `min(1,x)`) | `(pl, 1, secant(pl,pl)->(pu,1), x)` |
| lower knee | `pl < -1 ∧ pu ≤ 1` (convex `max(-1,x)`) | `(-1, pu, x, secant(pl,-1)->(pu,pu))` |
| both knees | `pl < -1 ∧ pu > 1` (clamp) | `(-1, 1, -1, 1)` |

**Soundness of the linear pieces.** On the single-knee cases the active branch is convex
(`max(-1,x)`) or concave (`min(1,x)`): a secant lies *above* a convex function (valid upper
bound) and *below* a concave function (valid lower bound); the identity line `x` is the exact
other side (`min(1,x) ≤ x`, `max(-1,x) ≥ x`). On the both-knees case the function is neither
convex nor concave, so we fall back to the sound box `L=-1, U=1` (`HardTanh ∈ [-1,1]`). The
Affine transformer is kept from `deeppoly.cf`. `provesound` can check this because HardTanh is
piecewise linear (it cannot check Sigmoid/Tanh). Expected: `Proved Affine` and `Proved HardTanh`.

The `hardtanh.cf` source:

In [7]:
print(open("hardtanh.cf").read())

// hardtanh.cf : a DeepPoly transformer for HardTanh(x) = max(-1, min(1, x)).
//
// HardTanh is piecewise linear with knees at x=-1 and x=1, and is monotone
// increasing, so the concrete bounds are l=HardTanh(prev[l]), u=HardTanh(prev[u]).
// The polyhedral bounds L,U are sound linear bounds over [prev[l], prev[u]]:
//
//   * flat-low  (pu <= -1)            : HardTanh == -1              -> (-1,-1,-1,-1)
//   * flat-high (pl >=  1)            : HardTanh ==  1              -> ( 1, 1, 1, 1)
//   * linear    (pl>=-1 and pu<=1)    : HardTanh == x              -> (pl,pu, x, x)
//   * upper knee(pl>=-1 and pu> 1)    : concave min(1,x) on [pl,pu]
//         lower L = secant (pl,pl)->(pu,1)   (secant is below a concave fn)
//         upper U = x                        (min(1,x) <= x)
//   * lower knee(pl< -1 and pu<=1)    : convex  max(-1,x) on [pl,pu]
//         lower L = x                        (max(-1,x) >= x)
//         upper U = secant (pl,-1)->(pu,pu)  (secant is above a convex fn)
//  

### Soundness with `provesound`

In [8]:
provesound("hardtanh.cf")

Proved Affine
Proved HardTanh
Provesound result: None


---
## Summary

| Part | Deliverable | Result |
|---|---|---|
| (a) | New certifier `dp2.cf` (DeepPoly + interval refinement); also `dp2_dualbound.cf` (keeps both ReLU lower bounds) | sound by construction |
| (b) | `provesound dp2.cf` / `dp2_dualbound.cf` | **Proved Affine, Proved Relu** |
| (c) | DeepPoly vs `dp2`, 100 images × 5 ε | **identical** (99/94/83/49/2) — explained above |
| (d) | `HardTanh` transformer | **Proved Affine, Proved HardTanh** |

`.cf` files are on GitHub in this repository under `q5/`.